In [132]:
from pulp import *
import pandas as pd

# masses in dalton
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342

tail_5prime = [d_ribose, oxygen, 2*oxygen + phosphorus, oxygen]

masses = pd.read_csv("../masses_all.tsv", sep="\t").set_index("nucleoside", drop=False)
# dbg:
# masses = masses.loc[["A", "C", "G", "U"]]

In [133]:
masses

,nucleoside,monoisotopic_mass
nucleoside,,
A,A,267.09675
G,G,283.09167
C,C,243.08552
U,U,244.06954
1A,1A,281.11240
...,...,...
10G,10G,409.15970
102G,102G,425.15470
105G,105G,538.20230


In [134]:
import random, re
from scipy.stats import norm

nucleoside_re = re.compile(r"\d*[ACGU]")

true_sequence = nucleoside_re.findall("CGGUU")
print(true_sequence)
n_fragments = 20

# sample random fragments from true sequence
fragment_noise = norm.rvs(scale=0.5, size=n_fragments)
def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l + 1, len(true_sequence))
    return l, r
fragments = [random_fragment() for _ in range(n_fragments)]
fragment_masses = [max(sum(masses.loc[b, "monoisotopic_mass"] for b in true_sequence[l:r]) + noise, 0.0) for noise, (l, r) in zip(fragment_noise, fragments)]

start_fragments = [i for i in range(n_fragments) if fragments[i][0] == 0]
end_fragments = [i for i in range(n_fragments) if fragments[i][1] == len(true_sequence)]

fragments, fragment_masses, fragment_noise

['C', 'G', 'G', 'U', 'U']


([(0, 3),
  (2, 4),
  (2, 5),
  (2, 3),
  (2, 4),
  (0, 2),
  (1, 2),
  (4, 5),
  (1, 4),
  (4, 5),
  (3, 5),
  (1, 3),
  (3, 5),
  (0, 4),
  (0, 4),
  (1, 3),
  (1, 3),
  (2, 4),
  (4, 5),
  (4, 5)],
 [809.1733012972356,
  526.480394768297,
  771.9991806141064,
  283.0319408490183,
  526.60464109596,
  526.1238584453527,
  284.0103856551938,
  243.7566713638013,
  810.0888514910392,
  242.84453281686936,
  487.5327021239612,
  566.30281052416,
  488.4379116670319,
  1053.0828047793657,
  1054.1147463840855,
  566.2367182495235,
  566.3996229789287,
  526.8472050504822,
  243.46821807144758,
  244.0002173496501],
 array([-0.0955587 , -0.68081523,  0.76843061, -0.05972915, -0.5565689 ,
        -0.05333155,  0.91871566, -0.31286864, -0.16402851, -1.22500718,
        -0.60637788,  0.11947052,  0.29883167, -0.25559522,  0.77634638,
         0.05337825,  0.21628298, -0.31400495, -0.60132193, -0.06932265]))

In [135]:
def estimate_seq(masses):
    prob = LpProblem("RNA sequencing", LpMinimize)
    # i = 1,...,n: positions in the sequence
    # j = 1,...,m: fragments
    # b = 1,...,k: (modified) bases

    n = len(true_sequence)
    m = len(fragment_masses)
    bases = masses.index.values

    # x: binary variables indicating fragment j presence at position i
    x = [[LpVariable(f"x_{i},{j}", lowBound=0, upBound=1, cat=LpInteger) for j in range(m)] for i in range(n)]
    # y: binary variables indicating base at position i
    y = [{b: LpVariable(f"y_{i},{b}", lowBound=0, upBound=1, cat=LpInteger) for b in bases} for i in range(n)]
    # z: binary variables indicating product of x and y
    z = [[{b: LpVariable(f"z_{i},{j},{b}", lowBound=0, upBound=1, cat=LpInteger) for b in bases} for j in range(m)] for i in range(n)]
    # weight_diff_abs: absolute value of weight_diff
    weight_diff_abs = [LpVariable(f"weight_diff_abs_{j}", lowBound=0, cat=LpContinuous) for j in range(m)]
    # weight_diff: difference between fragment monoisotopic mass and sum of masses of bases in fragment as estimated in the MILP
    weight_diff = [fragment_masses[j] - lpSum([z[i][j][b] * masses.loc[b, "monoisotopic_mass"] for i in range(n) for b in bases]) for j in range(m)]

    # optimization function
    prob += lpSum([weight_diff_abs[j] for j in range(m)])

    # select one base per position
    for i in range(n):
        
        prob += lpSum([y[i][b] for b in bases]) == 1

    # fill z with the product of binary variables x and y
    for i in range(n):
        for j in range(m):
            for b in bases:
                prob += z[i][j][b] <= x[i][j]
                prob += z[i][j][b] <= y[i][b]
                prob += z[i][j][b] >= x[i][j] + y[i][b] - 1

    # ensure that fragment is aligned continuously
    # (no gaps: if x[i,j] = 1 and x[i+2,j] = 1, then x[i+1,j] = 1)
    for j in range(m):
        for i in range(n - 2):
            prob += x[i][j] + x[i + 2][j] - 1 <= x[i + 1][j]

    # ensure that start fragments are aligned at the beginning of the sequence
    for j in start_fragments:
        x[0][j].setInitialValue(1)
        x[0][j].fixValue()

    # ensure that end fragments are aligned at the end of the sequence
    for j in end_fragments:
        x[n - 1][j].setInitialValue(1)
        x[n - 1][j].fixValue()

    # constrain weight_diff_abs to be the absolute value of weight_diff
    for j in range(m):
        prob += weight_diff_abs[j] >= weight_diff[j]
        prob += weight_diff_abs[j] >= -weight_diff[j]
    
    import pulp
    gurobi = pulp.getSolver("GUROBI_CMD")
    gurobi.msg = False
    result_ = prob.solve(gurobi)

    def get_base(i):
        for b in bases:
            if y[i][b].value() == 1:
                return b
        return None

    return [get_base(i) for i in range(n)]
    

In [136]:
from collections import defaultdict
from sklearn.cluster import KMeans
import numpy as np

reduce_to_k = 6

def reduce_alphabet(masses):
    kmeans = KMeans(n_clusters=reduce_to_k, random_state=213126)
    kmeans.fit(masses["monoisotopic_mass"].values.reshape(-1, 1))
    pseudo_nucs = [f"n{k}" for k in range(reduce_to_k)]
    nuc_mapping = defaultdict(list)
    for nuc, label in zip(masses["nucleoside"], kmeans.labels_):
        nuc_mapping[pseudo_nucs[label]].append(nuc)
    return (
        pd.DataFrame({
            "nucleoside": pseudo_nucs,
            "monoisotopic_mass": kmeans.cluster_centers_[:, 0]
        }).set_index("nucleoside", drop=False),
        nuc_mapping
    )

In [137]:
def expand_alphabet(seq, nuc_mapping):
    return {nuc for pseudo_nuc in seq for nuc in nuc_mapping[pseudo_nuc]}

In [138]:
start_fragments, end_fragments

([0, 5, 13, 14], [2, 7, 9, 10, 12, 18, 19])

In [139]:
current_masses = masses
seq = None

while True:
    print(f"remaining masses: {current_masses.shape[0]}")
    if current_masses.shape[0] > reduce_to_k:
        pseudo_masses, nuc_mapping = reduce_alphabet(current_masses)
        seq = estimate_seq(pseudo_masses)
        print(seq, nuc_mapping)
    else:
        seq = estimate_seq(current_masses)
        break
    current_masses = current_masses.loc[list(expand_alphabet(seq, nuc_mapping))]
seq

remaining masses: 67


/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['n5', 'n4', 'n5', 'n5', 'n4'] defaultdict(<class 'list'>, {'n4': ['A', 'G', 'C', 'U', '1A', '2A', '0A', '8A', '6A', '19A', '09A', '67A', '01A', '28A', '06A', '66A', '019A', '68A', '1G', '0G', '2G', '7G'], 'n0': ['64A', '066A', '621A', '61A', '60A', '01G', '02G', '22G', '27G', '4G', '022G', '027G', '227G', '42G', '34G', '342G', '100G', '101G', '103G'], 'n3': ['65A', '2161A', '69A', '2160A', '62A', '63A', '662A', '2165A', '2164A', '47G', '347G', '10G', '102G'], 'n1': ['2162A', '2163A', '00A', '348G', '3470G', '3480G', '00G'], 'n5': ['3483G', '34830G', '34832G', '105G'], 'n2': ['104G', '106G']})
remaining masses: 26


/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['n1', 'n3', 'n3', 'n0', 'n0'] defaultdict(<class 'list'>, {'n3': ['A', '8A', '6A', '1A', '09A', '19A', '0A', 'G', '2A'], 'n2': ['01A', '0G', '67A', '019A', '68A', '2G', '06A', '66A', '1G', '28A', '7G'], 'n1': ['34830G'], 'n4': ['105G', '34832G'], 'n5': ['3483G'], 'n0': ['C', 'U']})
remaining masses: 12


/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['n4', 'n4', 'n2', 'n2', 'n2'] defaultdict(<class 'list'>, {'n3': ['A'], 'n2': ['C', 'U'], 'n0': ['8A', '0A', '1A', '6A', '2A'], 'n1': ['34830G'], 'n5': ['09A', '19A'], 'n4': ['G']})
remaining masses: 3


/home/johannes/micromamba/envs/pystats/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


['C', 'G', 'G', 'U', 'U']